# MITRE ATT&CK → Automated MulVAL Rule Generator (MA&AM RuleGen)

This toolkit automates the process of aligning a MulVAL knowledge base with a pinned MITRE ATT&CK technique snapshot using an NLP/LLM pipeline.
The checked-in rule artifacts use MITRE ATT&CK v17.1 from upstream commit `d4a34a19eb60dcd0a9d15a456da842a42e1003fc`. The corresponding CSV snapshot was first recorded in Hydra's private history on October 19, 2025.

---

**Disclaimer: Use of genAI/LLMs**

All input data used in this research is obtained from an official, publicly maintained JSON database.
Data retrieval is performed via a controlled crawler that accessed only authorized endpoints, and the retrieved data is stored in version-controlled JSON files to ensure transparency and reproducibility.

AI use is inherent to this research. The research employs a hybrid processing pipeline integrating deterministic parsers, LLM-assisted NLP components, and human inspection.
Most preprocessing and transformation steps, such as schema normalization and rule structuring, are performed by deterministic parsers.
These parsers were initially generated with assistance from a coding agent, which produced prototype regular expressions and code fragments. 

An initial predicate list, used to guide rule induction, was also created with LLM assistance based on the textual database.
This list served as an exemplary input for subsequent stages of the pipeline. Although the LLM generated the first draft, these initial predicates were manually reviewed, corrected, and validated (face balidity) before use. 

This process exemplifies a rapid prototyping approach, in which model-assisted generation accelerates development while human inspection is performed for face validity. But: Natural language is ambugious, humans have their subjective judgements and make mistakes. 

In the natural-language processing stage, an LLM is employed exclusively for semantic interpretation and rule hypothesis generation.
At no point does the LLM operate autonomously or contribute unverified results. All outputs are to be critically evaluated, with users/researchers/engineers responsible for acceptance, modification, or rejection of candidate rules. So far, the methodology was inspection and face validiy check by the researchers (which is limited; rigorous evaluation is deferred for future work.)

The study's objective is methodological — to evaluate a reproducible framework for dynamic rule induction and contextual inference — rather than to assert the definitive correctness of any specific rule instance. Natural language is ambugious, formal rules are not (cf. Chomsky hierarchy); there is no lossless compression.

The LLM is treated as a computational instrument rather than a source of factual authority.

This methodological setup adheres to the scientific-ethical principles:

- Transparency – all data sources, code origins, and LLM-assisted components are documented;
- Reproducibility – datasets, sources and scripts are stated and versions included when possible; 
- HITL – human inspection is applied to AI-assisted output; researchers/users; 
  
LLMs are used to assist in two early stages of pipeline development: (1) generating a preliminary predicate list and (2) prototyping parser scripts.
All LLM-generated content was manually reviewed, and face validated before use.

Subsequent NLP steps also employ an LLM for controlled semantic interpretation as an additional check. 


---


## MITRE ATT&CK → Automated MulVAL Rule Generator Pipeline - Overview

### 🧩 1. Extract MITRE ATT&CK Techniques
**Script:** `extract_attack_techniques.py`

This script downloads the Enterprise, ICS or Mobile ATT&CK STIX JSON dataset and extracts all techniques and sub-techniques into a structured CSV file.

### 🕵️‍♂️ 2. Create MulVAL Predicate Set and Reference (MITRE ATT&CK Integration)

Has to be specified by user/system engineer. This serves as input for the AI-Agent.

**File:** `predicates.P`

### ⚙️ 3. Generate MulVAL Rule Stubs

`generate_mulval_rules.py` reads one or more per-domain techniques*.csv files (the ones produced in the first step extractor) and emits one .rule file per technique into a target directory. This is done using a parser. 

###  💻 4. Create Data Structures for NLP-Pipeline

##### 📈 Approach: Incremental Input Processing

* **Input:** Natural Language Rule Description <-- in Rule stubs (done)
* **Define Structure** Predicate Identification (initial predicate base); following initial Variable Extraction, Bound/Unbound Variable Classification
* **Fill Data Structurees:** Variable Extraction, Bound/Unbound Variable Classification, Rule creation; Predicate Updates

##### 🔄 Initialize 1 - Predicate table (global, canonical)

Builds initial core and meta predicate JSONL from `initial_predicates.P`. **Outputs:** `predicates_core.jsonl`, `predicates_meta.jsonl`

##### 🔄 Initialize 2 - Variable–Argument Canonicalization record (Initialization from Predicates)

This step builds a global variable concept dictionary (**Output:** `var_canonical.jsonl`) directly from the predicate registries (`predicates_core.jsonl`, `predicates_meta.jsonl`). Each record represents a semantic concept (like host, user, port, network) derived from argument roles defined in your predicates.
It serves as the single project-wide vocabulary for consistent variable naming across rules and LLM-generated content. This file becomes the naming and concept reference for your LLM rule generation pipeline, ensuring every new or updated rule stays semantically aligned with the project's existing fact base. 

### ✍️ 5. LLM-assisted Rule Creation/Updating from Natural Language Attack Descriptions 

**📝 Inputs:**

**Initial/Incremental Predicate and Variable Base**
`PRED_CORE_JSONL = "nlp_inputs/predicates_core.jsonl"`    # core domain predicates
`PRED_META_JSONL = "nlp_inputs/predicates_meta.jsonl"`    # meta layer predicates
`VAR_CANON_JSONL = "nlp_inputs/var_canonical.jsonl"`      # global variable concept dictionary

** LLM-Pipeline**
`AGENT_PROMPT = "nlp_inputs/systemprompt.md"`          # system prompt template

##### 🤖 Natural Language Processing Pipeline: Variable Extraction, Bound/Unbound Variable Classification, Rule creation

- 1. The script scans a directory (recursively) for *.rule files, extracts the content.
- 2. LLM Pipeline: the an LLM is called and presented with a system prompt: rule the *.rule file, and nlp_inputs/predicates_core.jsonl, nlp_inputs/predicates_meta.jsonl and nlp_inputs/var_canonical.jsonl as context and creates the rules.
- 3. the *.rule file is copied to nlp_outputs/ and the rule is updated. 
- 4. If necessary, the script adds new variables and predicates to nlp_inputs/var_canonical.jsonl and nlp_inputs/predicates_core.jsonl

### Pipeline Summary

This pipeline constructs `full_interaction_rules.P` through a deterministic assembly stage driven by LLM-assisted rule drafting. First, MITRE ATT&CK techniques are extracted from STIX and converted into per-technique `.rule` stubs, while the predicate baseline is initialized in `nlp_inputs/predicates_core.jsonl` (plus meta and variable-canonical registries). Next, an NLP stage updates each technique file with candidate Prolog clauses bounded by `% ---- RULE LLM START`/`% ---- RULE LLM END`, and optionally contributes new primitive declarations in `nlp_outputs/additional_predicates_from_llm.P`. The script `generate_full_kb.py` then performs the final compilation: it loads core and additional primitive predicates, scans all `nlp_outputs/*.rule`, extracts only marked rule blocks, parses clause heads/bodies to infer predicate signatures, and distinguishes primitive from derived symbols. The compiler writes a single MulVAL knowledge base at `kb/full_interaction_rules.P` containing (i) primitive declarations, (ii) fixed and auto-discovered `derived(...)` declarations, (iii) tabling directives for core attack-state predicates (`exec/2`, `compromise/2`, `priv_escalation/3`, `loot/3`, `exfiltrated/3`, `persistence/2`, `c2_channel/2`) plus additional inferred/recursive predicates to stabilize reasoning, (iv) utility and compatibility predicates (`member/2`, `append/3`, `reverse/2`, `file_extension/2`, protocol-arity shims), (v) raw technique clauses, and (vi) MulVAL-compatible `interaction_rule/2` wrappers (`rule_desc(label, metric)`) generated for selected derived heads. In effect, the artifact is a reproducible hybrid KB where ATT&CK semantics are captured in per-technique rules, while final graph-ready interaction logic is normalized, typed, and execution-safe during compilation.

---

## MITRE ATT&CK → MulVAL Rule Generator (MA&AM RuleGen)

This toolkit automates the process of aligning a MulVAL knowledge base with the pinned MITRE ATT&CK v17.1 technique snapshot.

---

# 🧩 1. Extract MITRE ATT&CK Techniques
**Script:** `extract_attack_techniques.py`

This script downloads the Enterprise, ICS or Mobile ATT&CK STIX JSON dataset and extracts all techniques and sub-techniques into a structured CSV file.

✅ **Features**
* Pulls directly from the official MITRE GitHub source
* Extracts:
    * Technique ID (e.g., T1190, T1566.001)
    * Name, description
    * Tactics (kill chain phase)
    * Platforms, detection text, and reference URL
* Outputs a clean, ready-to-use `techniques.csv` file

🚀 **Usage**

```bash
python3 extract_attack_techniques.py
```

📦 **Output**

`techniques.csv`
* `tid`
* `is_subtechnique`
* `name`
* `description`
* `platforms`
* `tactics`
* `detection`
* `data_sources`
* `url`
* `stix_id`

**Example:**

```
T1190, no, Exploit Public-Facing Application, "Adversaries may exploit...", Windows;Linux, initial-access, ...
```

---

⚙️ **2. Generate MulVAL Rule Stubs**
**Script:** `generate_mulval_rules.py` (under development)

This script reads `techniques.csv` and automatically creates one MulVAL rule file per MITRE ATT&CK technique. Each rule file is pre-filled with metadata headers and a generic logic template for the LLM to customize. The current rules are mere Stubs!

🏗️ **Example output structure**

```
mulval_rules/
├── T1190_exploit_public_service.rule
├── T1566_phishing.rule
└── T1550_creds_reuse.rule
```

🧱 **Example rule template**

```
% NOTE: Autogenerated stub. Adjust predicates/logic to your KB.
% technique_id: T0814
% technique_name: Denial of Service
% domain: ics
% tactic(s): inhibit-response-function
% platforms: None
% url: https://attack.mitre.org/techniques/T0814
% is_subtechnique: no
% stix_id: attack-pattern--1b22b676-9347-4c55-9a35-ef0dc653db5b
% description: Adversaries may perform Denial-of-Service (DoS) attacks to disrupt expected device functionality. Examples of DoS attacks include overwhelming the target device with a high volume of requests in a short time period and sending the target device a request it does not know how to handle. Disrupting device state may temporarily render it unresponsive, possibly lasting until a reboot can occur. When placed in this state, devices may be unable to send and receive requests, and may not perform expected response functions in reaction to other events in the environment.   Some ICS devices are particularly sensitive to DoS events, and may become unresponsive in reaction to even a simple ping sweep. Adversaries may also attempt to execute a Permanent Denial-of-Service (PDoS) against certain devic...
% generated_by: notebook_generate_rules

% ---- RULE STUB START (minimal) ----
potential_t0814(Host) :-
    attacker_at(net),
    connects(net, Host, _Port).

compromise(Host, user) :-
    potential_t0814(Host).
% ---- RULE STUB END ----

```

---


🔄 **Reproducing the Pinned Snapshot**

Run the extractor to reproduce the ATT&CK v17.1 CSV snapshot:

```bash
python3 extract_attack_techniques.py
```

Then regenerate your rule stubs:

```bash
python3 generate_mulval_rules.py
```


---

📚 **References**

* MITRE ATT&CK STIX Data: [https://github.com/mitre-attack/attack-stix-data](https://github.com/mitre-attack/attack-stix-data)
* MulVAL Project: [https://github.com/mulval/mulval](https://github.com/mulval/mulval)

---

⚙️ **To Dos / Issues**
* add Parser Argument --mobile / --ics/ --enterprise / --all for updates


---



### 🧩 1. Extract MITRE ATT&CK Techniques
**Script:** `extract_attack_techniques.py`

In [55]:
#!/usr/bin/env python3

"""
MITRE ATT&CK Technique Extraction Utility

Fetches the pinned MITRE ATT&CK v17.1 JSON data and converts it into a structured CSV format
for further processing. Supports Enterprise, Mobile, and ICS attack domains.

Features:
    - Retrieves technique data from official MITRE ATT&CK GitHub repository
    - Extracts key technique metadata
    - Generates CSV with normalized technique information
    - Supports main techniques and sub-techniques
    - Handles multiple attack domains (Enterprise/Mobile/ICS)

Configuration:
    - RAW_URL: Direct link to MITRE ATT&CK JSON data
    - OUT_CSV: Output CSV filename for extracted techniques

Extracted Technique Metadata:
    - Technique ID (TID)
    - Name
    - Description
    - Platforms
    - Tactics
    - Detection Hints
    - Data Sources
    - MITRE URL
    - STIX Identifier

Key Functions:
    - get_tid(): Extracts technique identifier from external references
    - flatten_kill_chain(): Normalizes tactic information
    - safe_text(): Cleans and sanitizes text fields
    - main(): Primary extraction and CSV generation logic

Usage:
    1. Uncomment desired RAW_URL and OUT_CSV for target domain
    2. Run script to generate technique CSV
    3. Use generated CSV with rule generation tools

Requirements:
    - requests
    - csv
    - html (for entity unescaping)

Example Domains:
    - Enterprise Attack: enterprise-attack.json
    - Mobile Attack: mobile-attack.json
    - ICS Attack: ics-attack.json

Note:
    - Requires internet access to fetch the pinned MITRE ATT&CK v17.1 data
    - Techniques are sorted by numeric TID for consistent output
    - Sub-techniques identified by dot notation in TID (e.g., T1234.001)

Dependencies:
    - Python 3.6+
    - External libraries: requests

Recommended Workflow:
    1. Extract techniques
    2. Generate rule stubs
    3. Manually refine generated rules or use NLP-LLM Pipeline
"""


import requests
import csv
import html
from urllib.parse import urljoin

RAW_URL = "https://raw.githubusercontent.com/mitre-attack/attack-stix-data/d4a34a19eb60dcd0a9d15a456da842a42e1003fc/enterprise-attack/enterprise-attack-17.1.json"
OUT_CSV = "enterprise-techniques.csv"

#RAW_URL = "https://raw.githubusercontent.com/mitre-attack/attack-stix-data/d4a34a19eb60dcd0a9d15a456da842a42e1003fc/mobile-attack/mobile-attack-17.1.json"
#OUT_CSV = "mobile-techniques.csv"

#RAW_URL = "https://raw.githubusercontent.com/mitre-attack/attack-stix-data/d4a34a19eb60dcd0a9d15a456da842a42e1003fc/ics-attack/ics-attack-17.1.json"
#OUT_CSV = "ics-techniques.csv"


def get_tid(obj):
    # find the external reference from mitre-attack that contains the TID
    for ref in obj.get("external_references", []):
        if ref.get("source_name") == "mitre-attack" and ref.get("external_id"):
            return ref.get("external_id"), ref.get("url", "")
    # fallback: try any external_reference with external_id
    for ref in obj.get("external_references", []):
        if ref.get("external_id"):
            return ref.get("external_id"), ref.get("url", "")
    return "", ""

def flatten_kill_chain(obj):
    phases = []
    for k in obj.get("kill_chain_phases", []):
        # STIX uses phase_name for tactic (mitre-attack kill chain)
        phase = k.get("phase_name")
        if phase:
            phases.append(phase)
    return ";".join(sorted(set(phases)))

def safe_text(s):
    if not s:
        return ""
    # strip newlines and HTML entities minimally
    t = " ".join(str(s).splitlines())
    return html.unescape(t).strip()

def main():
    print("Downloading ATT&CK JSON...")
    r = requests.get(RAW_URL, timeout=30)
    r.raise_for_status()
    data = r.json()

    techniques = []
    for o in data.get("objects", []):
        if o.get("type") == "attack-pattern":
            tid, url = get_tid(o)
            name = safe_text(o.get("name"))
            desc = safe_text(o.get("description") or o.get("x_mitre_description") or "")
            platforms = ";".join(o.get("x_mitre_platforms", [])) if o.get("x_mitre_platforms") else ""
            tactics = flatten_kill_chain(o)
            detection = safe_text(o.get("x_mitre_detection", ""))
            data_sources = ";".join(o.get("x_mitre_data_sources", [])) if o.get("x_mitre_data_sources") else ""
            # Include whether this looks like a sub-technique by checking for dot in external_id (Txxxx.yyy)
            is_sub = "." in tid if tid else False

            techniques.append({
                "tid": tid,
                "is_subtechnique": "yes" if is_sub else "no",
                "name": name,
                "description": desc,
                "platforms": platforms,
                "tactics": tactics,
                "detection": detection,
                "data_sources": data_sources,
                "url": url,
                "stix_id": o.get("id", "")
            })

    # sort by tid (puts main techniques and sub-techniques together)
    def sort_key(x):
        # empty TIDs go last
        if not x["tid"]:
            return ("Z", x["name"])
        # split numeric part for natural sort, keep subtech ordering
        parts = x["tid"].split(".")
        try:
            main = int(parts[0].lstrip("T"))
        except Exception:
            main = 999999
        sub = int(parts[1]) if len(parts) > 1 and parts[1].isdigit() else 0
        return (main, sub, x["tid"])

    techniques.sort(key=sort_key)

    print(f"Writing {len(techniques)} techniques to {OUT_CSV}...")
    fieldnames = ["tid", "is_subtechnique", "name", "description", "platforms", "tactics", "detection", "data_sources", "url", "stix_id"]
    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for t in techniques:
            writer.writerow(t)

    print("Done. Example: head -n 5 " + OUT_CSV)

if __name__ == "__main__":
    main()

Writing 823 techniques to enterprise-techniques.csv...
Done. Example: head -n 5 enterprise-techniques.csv


---

# 🕵️‍♂️ 2. Create MulVAL Predicate Set and Reference (MITRE ATT&CK Integration)

Has to be specified by user/system engineer.

## Original Predicates

```prolog

% Under GNU General Public License, see <http://www.gnu.org/licenses/>.

/******************************************************/
/****         Predicates Declaration              *****/
/******************************************************/

primitive(inCompetent(_principal)).
primitive(competent(_principal)).
primitive(clientProgram(_host, _programname)).
primitive(vulExists(_host, _vulID, _program)).
primitive(vulProperty(_vulID, _range, _consequence)).
primitive(hacl(_src, _dst, _prot, _port)).
primitive(attackerLocated(_host)).
primitive(hasAccount(_principal, _host, _account)).
primitive(networkServiceInfo(_host, _program, _protocol, _port, _user)).
primitive(setuidProgramInfo(_host, _program, _owner)).
primitive(nfsExportInfo(_server, _path, _access, _client)).
primitive(nfsMounted(_client, _clientpath, _server, _serverpath, _access)).
primitive(localFileProtection(_host, _user, _access, _path)).
primitive(dependsOn(_h, _program, _library)).
primitive(installed(_h, _program)).
primitive(bugHyp(_,_,_,_)).
primitive(vulExists(_machine,_vulID,_program,_range,_consequence)).
primitive(canAccessFile(_host, _user, _access, _path)).
primitive(isWebServer(_host)).
meta(cvss(_vulID, _ac)).


derived(execCode(_host, _user)).
derived(netAccess(_machine,_protocol,_port)).
derived(canAccessHost(_host)).
derived(accessFile(_machine,_access,_filepath)).
derived(accessMaliciousInput(_host, _principal, _program)).
derived(principalCompromised(_victim)).
derived(dos(_host)).
derived(logInService(_host, _protocol, _port)).

meta(attackGoal(_)).
meta(advances(_, _)).

/******************************************************/
/****         Tabling Predicates                  *****/
/*   All derived predicates should be tabled          */
/******************************************************/

:- table execCode/2.
:- table netAccess/3.
:- table canAccessHost/1.
:- table canAccessFile/4.
:- table accessFile/3.
:- table principalCompromised/1.
:- table vulExists/5.
:- table logInService/3.

```

## Updated Predicates

For the parser note: a comment blockn applies to the next predicate that follows!

```prolog

/******************************************************/
/****         Core Predicates (Enterprise Domain)       *****/
/******************************************************/

% Declares a host or asset
primitive(host(_H)).

% Attacker has access to network segment Net
primitive(attacker_at(_Net)).

% Network path from Net to host H on Port
primitive(connects(_Net, _H, _Port)).

% Host Src can reach host Dst on Port
primitive(reachable(_Src, _Dst, _Port)).

% Network service running
primitive(service(_H, _Port, _Proto, _Svc, _Version)).

% Service is externally reachable
primitive(exposed(_H, _Port)).

% OS or platform
primitive(os(_H, _OS)).

% Asset role (app, db, dc, plc, etc.)
primitive(role(_H, _Role)).


/******************************************************/
/****         Software, Configuration & Vulnerabilities *****/
/******************************************************/

% Installed package
primitive(pkg(_H, _Name, _Version)).

% Config parameter
primitive(setting(_H, _Key, _Value)).

% Vulnerability exists on asset
primitive(vuln(_H, _CVE, _Artifact)).

% CVE has a working exploit
primitive(exploit_possible(_CVE)).

% CVE affects service
primitive(service_matches(_CVE, _Svc)).

% Local (priv-esc) vuln present
primitive(local_vuln(_H, _CVE)).

% Non-CVE weakness
primitive(misconfig(_H, _Issue)).


/******************************************************/
/****         Identity, Auth & Privilege              *****/
/******************************************************/

% User account exists
primitive(user_on_host(_User, _H)).

% Credential known/available
primitive(cred(_User, _Scope, _Type)).

% Host accepts auth method
primitive(auth_possible(_H, _User, _Method)).

% Pass-the-hash works
primitive(accepts_hash(_H, _User)).

% User privilege level
primitive(perm(_H, _User, _Priv)).

% Implicit trust or delegation
primitive(trust(_H1, _H2, _Mechanism)).


/******************************************************/
/****         Execution, Persistence & Defense Evasion  *****/
/******************************************************/

% Code executed as user
primitive(exec(_H, _User)).

% Host compromised at privilege level
primitive(compromise(_H, _Level)).

% Persistence established
primitive(persistence(_H, _Method)).

% Local privilege escalation
primitive(priv_escalation(_H, _From, _To)).

% Disabled defense
primitive(defense_disabled(_H, _Control)).

% Captured sensitive artifact
primitive(loot(_H, _Kind, _Id)).


/******************************************************/
/****         Collection, Exfiltration & Command and Control *****/
/******************************************************/

% Host holds sensitive data
primitive(has_data(_H, _DataKind)).

% Command-and-control channel established
primitive(c2_channel(_H, _Proto)).

% Data exfiltrated
primitive(exfiltrated(_H, _DataKind, _Channel)).


/******************************************************/
/****         Observations & Evidence                 *****/
/******************************************************/

% Observed event mapped to ATT&CK
primitive(event(_H, _Tactic, _TechniqueID, _EvidenceID)).

% Telemetry available
primitive(sensor(_H, _Type)).


/******************************************************/
/****         Cloud Extensions                        *****/
/******************************************************/

% Cloud account
primitive(cloud_account(_Acct, _Provider)).

% IAM principal
primitive(iam_principal(_Principal, _Acct)).

% IAM policy
primitive(iam_policy(_Principal, _Action, _Resource, _Effect)).

% Cloud credential
primitive(cloud_cred(_Principal, _Type)).

% Metadata access
primitive(metadata_access(_H, _Provider)).


/******************************************************/
/****         Mobile Extensions                       *****/
/******************************************************/

% Mobile device asset
primitive(device(_D)).

% App installed
primitive(app_installed(_D, _App, _Version)).

% Mobile permission
primitive(mobile_perm(_App, _Permission)).

% MDM enrollment
primitive(mdm_enrolled(_D, _Vendor)).


/******************************************************/
/****         ICS / OT Extensions                     *****/
/******************************************************/

% PLC asset
primitive(plc(_P)).

% ICS protocol
primitive(protocol(_P, _Proto, _Port)).

% PLC ladder logic
primitive(ladder_logic(_P, _Program)).

% Safety Instrumented System
primitive(safety_instrumented(_SIS)).


/*************************************************/


/******************/
/**            Derived Predicates               **/
/******************/

% --- Reachability shortcuts & closure ---

% External exposure from any attacker-controlled segment
derived(exposed(H, Port)).
exposed(H, Port) :-
    attacker_at(Net),
    connects(Net, H, Port).

% Base & transitive lateral reachability
derived(reachable(Src, Dst, Port)).
reachable(Src, Dst, Port) :-
    connects(Src, Dst, Port).

reachable(Src, Dst, Port) :-
    reachable(Src, Mid, Port),
    connects(Mid, Dst, Port).

% --- Vulnerability composition helpers ---

% A service instance is affected by an exploitable CVE
derived(remote_service_vulnerable(H, Port, CVE, Svc)).
remote_service_vulnerable(H, Port, CVE, Svc) :-
    service(H, Port, _Proto, Svc, _Version),
    vuln(H, CVE, Svc),
    service_matches(CVE, Svc),
    exploit_possible(CVE).

% Local privilege-escalation opportunity
derived(local_pe_vulnerable(H, CVE)).
local_pe_vulnerable(H, CVE) :-
    local_vuln(H, CVE),
    exploit_possible(CVE).

% --- Authentication & identity helpers ---

% Simple notion of "valid account" for a host/scope
derived(valid_account(User, Host)).
valid_account(User, Host) :-
    user_on_host(User, Host).

% Credentials that can be used for auth to a host
derived(auth_usable_cred(User, Host, Method)).
auth_usable_cred(User, Host, Method) :-
    cred(User, Host, _Type),
    auth_possible(Host, User, Method).

% Pass-the-hash feasibility
derived(pth_feasible(Host, User)).
pth_feasible(Host, User) :-
    cred(User, Host, hash),
    accepts_hash(Host, User).

% --- Execution & compromise enablers ---

% Remote code execution via exposed vulnerable service
derived(exec_possible_remote(H, User)).
exec_possible_remote(H, www_user) :-
    attacker_at(Net),
    connects(Net, H, Port),
    remote_service_vulnerable(H, Port, _CVE, _Svc).

% User-driven (phishing/SE) execution path (leave SE success as fact)
derived(exec_possible_user(H, User)).
exec_possible_user(H, User) :-
    user_on_host(User, H),
    auth_possible(H, User, _Method),
    user_phishes_success(User).

% Compromise follows from execution (conservative default)
derived(compromise_from_exec(H, Level)).
compromise_from_exec(H, user) :- exec(H, _AnyUser).

% Priv-esc derived effect
derived(compromise_from_pe(H, To)).
compromise_from_pe(H, To) :-
    priv_escalation(H, _From, To).

% --- Persistence & defense effects ---

derived(persistence_enables_reentry(H)).
persistence_enables_reentry(H) :-
    persistence(H, _Method).

derived(weakened_defense(H)).
weakened_defense(H) :-
    defense_disabled(H, _Control).

% --- ICS helpers ---

derived(ics_endpoint(P, Port)).
ics_endpoint(P, Port) :-
    plc(P),
    protocol(P, _Proto, Port).

derived(ics_exposed_endpoint(P, Port)).
ics_exposed_endpoint(P, Port) :-
    attacker_at(Net),
    connects(Net, P, Port),
    ics_endpoint(P, Port).


/******************/
/**            Canonical consequence rules         **/
/******************/

% Remote service exploit → execution → compromise
exec(H, www_user) :-
    attacker_at(Net),
    connects(Net, H, Port),
    remote_service_vulnerable(H, Port, _CVE, _Svc).

compromise(H, user) :-
    exec(H, _).

% User/social engineering path
exec(H, User) :-
    exec_possible_user(H, User).

compromise(H, user) :-
    exec_possible_user(H, _).

% Local privilege escalation
priv_escalation(H, From, To) :-
    exec(H, _),
    local_pe_vulnerable(H, _CVE),
    % optionally require current privilege = From
    nonvar(From), nonvar(To).

compromise(H, To) :-
    priv_escalation(H, _From, To).

% Pass-the-hash lateral move
exec(Dst, User) :-
    pth_feasible(Dst, User),
    reachable(Src, Dst, 445).

compromise(Dst, user) :-
    pth_feasible(Dst, _),
    reachable(_Src, Dst, 445).

% ICS exposed endpoint (placeholder consequence; refine per technique)
compromise(P, user) :-
    ics_exposed_endpoint(P, _Port).


/******************/
/**                  Meta Annotations              **/
/******************/
% Machine-readable metadata to keep the KB lean and traceable.

% technique(TID, Name, Domain, Tactics, Platforms, URL, STIX_ID).
meta(technique(_Tid, _Name, _Domain, _Tactics, _Platforms, _Url, _StixId)).

% severity(TID, Level) and confidence(TID, Level)
% Level ∈ {low, medium, high, critical}
meta(severity(_Tid, _Level)).
meta(confidence(_Tid, _Level)).

% data_source(TID, Source) and detection_hint(TID, Text)
meta(data_source(_Tid, _Source)).
meta(detection_hint(_Tid, _Text)).

% rule_map(TID, RuleTag) – points a technique to a canonical rule/consequence
% e.g., RuleTag ∈ {remote_exec, user_exec, priv_esc, pth_lateral, ics_exposed, persistence}
meta(rule_map(_Tid, _RuleTag)).

% grouping for fused techniques: fuse_id(FuseKey, Tid)
meta(fuse_id(_FuseKey, _Tid)).

% provenance: which file emitted a rule originally (useful before merging)
meta(source_file(_Tid, _Path)).

% Example meta facts (keep in a separate .meta if preferred):
% meta(technique('T0800','Activate Firmware Update Mode','ics','inhibit-response-function','None',
%                'https://attack.mitre.org/techniques/T0800',
%                'attack-pattern--19a71d1e-6334-4233-8260-b749cae37953')).
% meta(rule_map('T0800', ics_exposed)).
% meta(severity('T0800', high)).
% meta(confidence('T0800', medium)).
% meta(data_source('T0800', 'Network Traffic')).
% meta(detection_hint('T0800', 'Monitor unexpected transitions to firmware update mode on PLCs')).


/******************/
/**        Legacy Compatibility (Adapters)         **/
/******************/
% Uncomment or tweak as needed if you still ingest classic MulVAL facts.

% connects(Net,H,Port) from legacy hacl/attackerAt
% derived(connects(_Net, _H, _Port)).  % declaration for tooling
% connects(Net, H, Port) :-
%    attackerAt(Attacker),
%    hacl(Attacker, H, Port, tcp),
    % bind attacker segment to a symbolic Net (if you model segments)
%    Net = internet.

% service/5 from legacy networkServiceInfo/5
% derived(service(_H, _P, _Proto, _App, _Ver)).
% service(H, P, tcp, App, Ver) :-
%    networkServiceInfo(H, P, tcp, App, Ver).

% vuln/3 from legacy vulExists/4
% derived(vuln(_H, _CVE, _Artifact)).
% vuln(H, CVE, App) :-
%    vulExists(H, App, CVE, App).

% compromise/2 from execCode/2
% derived(compromise(_H, _Priv)).
% compromise(H, Priv) :-
%    execCode(H, Priv).


/******************/
/**            Tabling directives (XSB/MulVAL)    **/
/******************/

% NOTE:
% - Keep these BEFORE the corresponding predicate definitions.
% - If your engine supports incremental tabling (XSB ≥3.8) and you
%   update base facts at runtime, you can switch to the as incremental
%   variants shown below (commented).

:- table exposed/2.
:- table reachable/3.
:- table remote_service_vulnerable/4.
:- table local_pe_vulnerable/2.
:- table valid_account/2.
:- table auth_usable_cred/3.
:- table pth_feasible/2.
:- table exec_possible_remote/2.
:- table exec_possible_user/2.
:- table compromise_from_exec/2.
:- table compromise_from_pe/2.
:- table persistence_enables_reentry/1.
:- table weakened_defense/1.
:- table ics_endpoint/2.
:- table ics_exposed_endpoint/2.

% If you also define these via rules in the same KB, table them too to
% break cycles and cache results (recommended):
:- table exec/2.
:- table compromise/2.
:- table priv_escalation/3.

% -------- Optional (XSB) incremental variants --------
% Uncomment these instead of the plain :- table ... lines if you want
% derived answers to re-compute automatically when base facts change.
% :- table exposed/2               as incremental.
% :- table reachable/3             as incremental.
% :- table remote_service_vulnerable/4 as incremental.
% :- table local_pe_vulnerable/2   as incremental.
% :- table valid_account/2         as incremental.
% :- table auth_usable_cred/3      as incremental.
% :- table pth_feasible/2          as incremental.
% :- table exec_possible_remote/2  as incremental.
% :- table exec_possible_user/2    as incremental.
% :- table compromise_from_exec/2  as incremental.
% :- table compromise_from_pe/2    as incremental.
% :- table persistence_enables_reentry/1 as incremental.
% :- table weakened_defense/1      as incremental.
% :- table ics_endpoint/2          as incremental.
% :- table ics_exposed_endpoint/2  as incremental.
% :- table exec/2                  as incremental.
% :- table compromise/2            as incremental.
% :- table priv_escalation/3       as incremental.
```


We defined a unified predicate vocabulary for representing attack surfaces, vulnerabilities, and adversary behaviors across 

* Enterprise,
* Mobile, and
* ICS environments.  

It serves as the canonical reference for all .rule files generated from the MITRE ATT&CK mapping workflow.

---

## 📘 Design Goals

- *Unified:* One consistent vocabulary across platforms and tactics 
- *Compact and Composable:* Small, orthogonal predicates you can combine in rules  
- *MulVAL-Compatible:* Provide adapters provided for legacy predicates (hacl, vulExists, execCode) to enable downward compatibility
- *Extensible:* Optional modules for Cloud, Mobile, and ICS  
- *Fact-Driven:* Keep rules pure logic; express evidence as facts (datalog) 

---

## 🏗️ Core Predicates (Enterprise Domain)

### 🕸️ Topology & Reachability

| Predicate | Meaning | Example |
|------------|----------|----------|
| host(H) | Declares a host or asset | host(web01). |
| attacker_at(Net) | Attacker has access to network segment Net | attacker_at(internet). |
| connects(Net, H, Port) | Network path from Net to host H on Port | connects(internet, web01, 80). |
| reachable(Src, Dst, Port) | Host Src can reach host Dst on Port (used for lateral movement) | reachable(web01, app01, 5432). |
| service(H, Port, Proto, Svc, Version) | A network service is running | service(web01, 80, tcp, httpd, "2.4.49"). |
| exposed(H, Port) | Service is externally reachable | exposed(web01, 443). |
| os(H, OS) | OS or platform | os(web01, linux). |
| role(H, Role) | Asset role (app, db, dc, plc, etc.) | role(db01, database). |

---

### 🧱 Software, Configuration & Vulnerabilities

| Predicate | Meaning | Example |
|------------|----------|----------|
| pkg(H, Name, Version) | Installed package | pkg(web01, openssl, "1.1.1k"). |
| setting(H, Key, Value) | Config parameter | setting(web01, "smb_signing", "disabled"). |
| vuln(H, CVE, Artifact) | Vulnerability exists on asset | vuln(web01, "CVE-2021-41773", httpd). |
| exploit_possible(CVE) | CVE has a working exploit | exploit_possible("CVE-2021-41773"). |
| service_matches(CVE, Svc) | CVE affects service | service_matches("CVE-2021-41773", httpd). |
| local_vuln(H, CVE) | Local (priv-esc) vuln present | local_vuln(win10, "CVE-2021-40449"). |
| misconfig(H, Issue) | Non-CVE weakness | misconfig(db01, "default_password"). |

---

### 👤 Identity, Auth & Privilege

| Predicate | Meaning | Example |
|------------|----------|----------|
| user_on_host(User, H) | User account exists | user_on_host(alice, web01). |
| cred(User, Scope, Type) | Credential known/available | cred(alice, web01, password). |
| auth_possible(H, User, Method) | Host accepts auth method | auth_possible(web01, alice, ssh). |
| accepts_hash(H, User) | Pass-the-hash works | accepts_hash(dc01, alice). |
| perm(H, User, Priv) | User privilege level | perm(dc01, alice, user). |
| trust(H1, H2, Mechanism) | Implicit trust or delegation | trust(web01, db01, api_token). |

---

### ⚙️ Execution, Persistence & Defense Evasion

| Predicate | Meaning | Example |
|------------|----------|----------|
| exec(H, User) | Code executed as user | exec(web01, www_user). |
| compromise(H, Level) | Host compromised at privilege level | compromise(web01, user). |
| persistence(H, Method) | Persistence established | persistence(web01, crontab). |
| priv_escalation(H, From, To) | Local privilege escalation | priv_escalation(web01, user, root). |
| defense_disabled(H, Control) | Disabled defense | defense_disabled(web01, selinux). |
| loot(H, Kind, Id) | Captured sensitive artifact | loot(web01, credential_dump, hash_file). |

---

### 📤 Collection, Exfiltration & Command and Control

| Predicate | Meaning | Example |
|------------|----------|----------|
| has_data(H, DataKind) | Host holds sensitive data | has_data(db01, customer_records). |
| c2_channel(H, Proto) | Command-and-control channel established | c2_channel(web01, https). |
| exfiltrated(H, DataKind, Channel) | Data exfiltrated | exfiltrated(db01, customer_records, https). |

---

### 🧩 Observations & Evidence (for detection mapping)

| Predicate | Meaning | Example |
|------------|----------|----------|
| event(H, Tactic, TechniqueID, EvidenceID) | Observed event mapped to ATT&CK | event(web01, execution, T1059, ev123). |
| sensor(H, Type) | Telemetry available | sensor(web01, sysmon). |

---

## 🔄 Compatibility Layer (Legacy MulVAL)

Bridge old-style predicates to the new ones:

```prolog
connects(Net, H, P) :- attackerAt(Att), hacl(Att, H, P, tcp).
service(H, P, tcp, App, Ver) :- networkServiceInfo(H, P, tcp, App, Ver).
vuln(H, CVE, App) :- vulExists(H, App, CVE, App).
compromise(H, Priv) :- execCode(H, Priv)
```


---

### Possible Extensions - Cloud, Mobile, and ICS 

These extensions provide predicates for modeling assets and capabilities within Cloud, Mobile, and Industrial Control Systems (ICS) environments. They are designed to be used with a core set of predicates (like `connects/3`, `service/5`, `cred/3`, `compromise/2`) to build comprehensive attack graphs and detection rules.

## ☁️ Cloud Extensions

These predicates represent assets and configurations within cloud environments (AWS, Azure, GCP).

**Predicates**

| Predicate Signature | Meaning |
|---|---|
| `cloud_account(Acct, Provider)` | Declares a cloud tenant/account. `Acct` is the account identifier, `Provider` is the cloud provider (`aws`, `azure`, or `gcp`). |
| `iam_principal(Principal, Acct)` | An IAM user/role/app exists in `Acct`. `Principal` is the identifier of the IAM principal, `Acct` is the account. |
| `iam_policy(Principal, Action, Resource, Effect)` | Policy rule for a principal. `Principal` is the IAM principal, `Action` is the permission granted, `Resource` is the resource the policy applies to, and `Effect` is `allow` or `deny`. |
| `cloud_cred(Principal, Type)` | Credential available for the principal.  `Principal` is the IAM principal, `Type` is the credential type (`access_key`, `role_token`, `refresh_token`, `oauth_token`). |
| `metadata_access(H, Provider)` | Host `H` can reach instance metadata service (SSRF/lateral risk). `Provider` is the cloud provider (`aws`, `azure`, or `gcp`). |

**Example Facts**

```prolog
cloud_account(dev_acct, aws).
iam_principal("ec2-backup-role", dev_acct).
iam_policy("ec2-backup-role", "s3:GetObject", "arn:aws:s3:::corp-data/*", allow).
cloud_cred("ec2-backup-role", role_token).
metadata_access(web01, aws).
```


**Notes**

* Map `iam_policy` to technique rules for privilege escalation (e.g., `PassRole`, wildcard `Action`, etc.).
* Use `metadata_access/2` to model SSRF-to-IMDS attack chains.

---

## 📱 Mobile Extensions

These predicates represent assets and configurations within mobile environments.

**Predicates**

| Predicate Signature | Meaning |
|---|---|
| `device(D)` | Declares a mobile device asset. `D` is the device identifier. |
| `app_installed(D, App, Version)` | App `App` with `Version` is present on device `D`. |
| `mobile_perm(App, Permission)` | App `App` holds `Permission` permission (e.g., `READ_CONTACTS`). |
| `mdm_enrolled(D, Vendor)` | Device `D` is enrolled in MDM/EMM solution `Vendor`. |

**Example Facts**

```prolog
device(phone01).
app_installed(phone01, "com.bank.app", "5.2.1").
mobile_perm("com.bank.app", "READ_CONTACTS").
mdm_enrolled(phone01, intune).
```


**Notes**

* Combine `app_installed + mobile_perm` with collection/exfil rules (e.g., contacts, SMS).
* `mdm_enrolled` can increase/limit attack paths depending on policy predicates you add.

---

## 🏭 ICS / OT Extensions

These predicates represent assets and configurations within Industrial Control Systems (ICS) environments.

**Predicates**

| Predicate Signature | Meaning |
|---|---|
| `plc(P)` | Declares a PLC asset. `P` is the PLC identifier. |
| `protocol(P, Proto, Port)` | Fieldbus/ICS protocol `Proto` exposed on PLC `P` at `Port`. `Proto` can be `modbus`, `iec104`, `dnp3`, or `opcua`. |
| `ladder_logic(P, Program)` | Control logic/program `Program` loaded on PLC `P`. |
| `safety_instrumented(SIS)` | Safety Instrumented System present. `SIS` is the identifier of the safety system. |

**Example Facts**

```prolog
plc(plc01).
protocol(plc01, modbus, 502).
ladder_logic(plc01, "mixing_v3").
safety_instrumented(sis_unit1).
```


**Notes**

* Pair `protocol/3` with `connects/3` to model unauthenticated protocol reachability (e.g., Modbus function codes).
* Use `ladder_logic/2` to gate rules that require write access to logic (persistence/sabotage techniques).

---

## ToDos:

* **Keep domains orthogonal:** Check using face validity.
* **Add derived helpers:** For example:
    ```
    imds_reachable(H) :- metadata_access(H, _).
    mobile_has_perm(D, Perm) :- app_installed(D, App, _), mobile_perm(App, Perm).
    ics_exposed(P) :- protocol(P, _,_), exposed(P, _Port).
    ```
    Can only be done after we have updated the initial rule base (in a couple of HITL-iterations)

---

# ⚙️ 3. Generate MulVAL Rule Stubs


`generate_mulval_rules.py` reads one or more per-domain techniques*.csv files (the ones produced in the first step extractor) and emits one .rule file per technique into a target directory.

It writes a rich metadata header (TID, name, domain, tactics, platforms, URL, description, STIX id) and fills in a MulVAL-style rule stub using the updated predicates from your spec (e.g., connects/3, service/5, vuln/3, exec/2, compromise/2, etc.). An auto template picks a sensible body based on domain and tactics, or you can force a specific template.

**ATTENTION: THESE ARE RAW RULE STUBS!!! CONTENT WILL BE OVERWORKED IM THE LLM PIPELINE (CURRENTLY MISSING)**

---


## What you get (Output)
* Clean filenames like:
    
```
mulval_rules/
  enterprise_T1190_exploit_public_facing_application.rule
  enterprise/subtechniques/enterprise_T1566_001_spearphishing_attachment.rule   # if --sep-subtech
  mobile_T1409_access_contacts.rule
  ics_T0865_modify_controller_logic.rule
```

* Headers aligned with your Updated Predicates 
* An optional rules.manifest.json for bookkeeping
 



### ⚙️ 3. Generate MulVAL Rule Stubs
**Script:** see below

In [49]:
# MulVAL rule template generator (minimal- STUBS to be filled by LLM)
# ------------------------------------------------------
# Set these and run:
# CSV_PATHS = ["enterprise-techniques.csv", "mobile-techniques.csv", "ics-techniques.csv"]
# OUT_DIR = "mulval_rules"
# generate_rules(CSV_PATHS, OUT_DIR, domain_prefix=True, sep_subtech=True, force=False)

"""
MulVAL Rule Template Generator

Generates rule stubs from technique CSV files for MulVAL rule system.

Parameters:
    csv_paths (List[str]): Paths to input CSV files containing technique definitions.
        Each CSV should contain columns like:
        - tid (Technique ID)
        - name (Technique Name)
        - domain (Technique Domain)
        - description (Technique Description)
        - is_subtechnique (Boolean flag)

    out_dir (str): Output directory for generated rule files.
        Will be created if it doesn't exist.

    domain_prefix (bool, optional): 
        - If True, prepends domain name to rule filenames.
        - Defaults to True.
        - Example: 'enterprise_T1234_technique_name.rule'

    sep_subtech (bool, optional):
        - If True, places subtechniques in a 'subtechniques/' subdirectory.
        - Defaults to True.

    force (bool, optional):
        - If True, overwrites existing rule files.
        - If False, skips generation for existing files.
        - Defaults to False.

    manifest_path (str, optional):
        - Path to output JSON manifest of generated rules.
        - Set to None or empty string to skip manifest generation.
        - Defaults to "rules.manifest.json".

Returns:
    Dict: A manifest of generated rules, with technique IDs/names as keys.

Example:
    generate_rules(
        csv_paths=["enterprise-techniques.csv"],
        out_dir="mulval_rules",
        domain_prefix=True,
        sep_subtech=True
    )

Dependencies:
    - csv
    - os
    - re
    - json
    - textwrap

Note:
    - Requires input CSVs with specific column structure
    - Generates minimal rule stubs to be further refined
"""


import csv, os, re, json, textwrap
from typing import List, Dict, Tuple

# ---------------------------
# Utilities
# ---------------------------
def _safe_filename(s: str) -> str:
    s = (s or "").strip().lower()
    s = re.sub(r"[^\w\.-]+", "_", s)   # replace non-safe chars with "_"
    s = re.sub(r"_+", "_", s)          # collapse multiple underscores
    return s.strip("_") or "unnamed"

def _ensure_dir(p: str) -> None:
    if p and not os.path.exists(p):
        os.makedirs(p, exist_ok=True)

def _make_rule_relpath(tid: str, name: str, domain: str,
                       domain_prefix: bool, is_sub: bool, sep_subtech: bool) -> str:
    base = f"{tid}_{_safe_filename(name)}" if tid else _safe_filename(name)
    if domain_prefix and domain:
        base = f"{domain}_{base}"
    fname = base + ".rule"
    if is_sub and sep_subtech:
        return os.path.join("subtechniques", fname)
    return fname

def _read_csv_rows(path: str) -> List[Dict]:
    with open(path, newline="", encoding="utf-8") as f:
        r = csv.DictReader(f)
        return [dict(x) for x in r]

# ---------------------------
# Metadata + single template
# ---------------------------
_HEADER_NOTE = "% NOTE: Autogenerated stub. Adjust predicates/logic to your KB.\n"

def _render_metadata(row: Dict) -> str:
    desc = (row.get("description") or "").replace("\r", " ").replace("\n", " ").strip()
    if len(desc) > 5000:
        desc = desc[:4997] + "..."
        print("[WARNING] Attack description shortened.")
    return textwrap.dedent(f"""\
    % technique_id: {row.get('tid','')}
    % technique_name: {row.get('name','')}
    % domain: {row.get('domain','')}
    % tactic(s): {row.get('tactics','')}
    % platforms: {row.get('platforms','')}
    % url: {row.get('url','')}
    % is_subtechnique: {row.get('is_subtechnique','no')}
    % stix_id: {row.get('stix_id','')}
    % description: {desc}
    % generated_by: notebook_generate_rules
    """)

def _tid_slug(row: Dict) -> str:
    base = row.get("tid") or row.get("name") or "technique"
    return _safe_filename(base).replace(".", "_")

def _tpl_minimal(row: Dict) -> str:
    slug = _tid_slug(row)
    return textwrap.dedent(f"""\
    % ---- RULE STUB START (minimal) ----
    potential_{slug}(Host) :-
        attacker_at(net),
        connects(net, Host, _Port).

    compromise(Host, user) :-
        potential_{slug}(Host).
    % ---- RULE STUB END ----
    """)

# ---------------------------
# Core generator (minimal-only)
# ---------------------------

def _extract_domain_from_filename(csv_path: str) -> str:
    """
    Extract domain from CSV filename
    
    Args:
        csv_path (str): Path to the CSV file
    
    Returns:
        str: Extracted domain (enterprise, mobile, or ics)
    """
    filename = os.path.basename(csv_path).lower()
    for domain in ['enterprise', 'mobile', 'ics']:
        if domain in filename:
            return domain
    return ""

def generate_rules(csv_paths: List[str],
                   out_dir: str,
                   domain_prefix: bool = True,
                   sep_subtech: bool = True,
                   force: bool = False,
                   manifest_path: str = "rules.manifest.json") -> Dict[str, Dict]:
    """
    Generate .rule files from one or more per-domain techniques CSVs.
    Always uses the same minimal rule stub for every technique.
    """
    _ensure_dir(out_dir)

    # Load rows
    all_rows: List[Dict] = []
    for p in csv_paths:
        if not os.path.exists(p):
            print(f"[WARN] CSV not found: {p}")
            continue
        rows = _read_csv_rows(p)
        if not rows:
            print(f"[WARN] No rows in: {p}")
        
        # Extract domain from filename
        domain = _extract_domain_from_filename(p)
        
        # Add domain to each row if not already present
        for row in rows:
            if not row.get('domain'):
                row['domain'] = domain
        
        all_rows.extend(rows)


    if not all_rows:
        raise RuntimeError("No techniques found in provided CSV(s).")

    # Sort for stable output
    def _sort_key(r: Dict) -> Tuple:
        domain = (r.get("domain") or "zzz").lower()
        tid = (r.get("tid") or "").upper()
        m = re.match(r"T(\d+)(?:\.(\d+))?$", tid)
        if m:
            main = int(m.group(1)); sub = int(m.group(2) or 0)
        else:
            main = 999999; sub = 0
        return (domain, main, sub, tid, (r.get("name") or ""))

    all_rows.sort(key=_sort_key)

    manifest: Dict[str, Dict] = {}

    for row in all_rows:
        tid = (row.get("tid") or "").strip()
        name = (row.get("name") or "unnamed").strip()
        domain = (row.get("domain") or "").strip().lower()
        is_sub = (row.get("is_subtechnique","no").lower() in ("yes","true","1"))

        relpath = _make_rule_relpath(tid, name, domain, domain_prefix, is_sub, sep_subtech)
        out_path = os.path.join(out_dir, relpath)
        _ensure_dir(os.path.dirname(out_path))

        if os.path.exists(out_path) and not force:
            print(f"[SKIP] {out_path} (exists)")
            manifest.setdefault(tid or name, {}).update({"file": relpath, "domain": domain, "name": name})
            continue

        meta = _render_metadata(row)
        body = _tpl_minimal(row)

        with open(out_path, "w", encoding="utf-8") as f:
            f.write(_HEADER_NOTE)
            f.write(meta)
            f.write("\n")
            f.write(body)

        print(f"[WRITE] {out_path}")
        manifest.setdefault(tid or name, {}).update(
            {"file": relpath, "domain": domain, "name": name}
        )

    if manifest_path:
        _ensure_dir(os.path.dirname(manifest_path) or ".")
        with open(manifest_path, "w", encoding="utf-8") as mf:
            json.dump(manifest, mf, indent=2, ensure_ascii=False)
        print(f"[INFO] Wrote manifest: {manifest_path}")

    print("[OK] Generation complete.")
    return manifest

# Example:
# CSV_PATHS = ["enterprise-techniques.csv", "mobile-techniques.csv", "ics-techniques.csv"]
# OUT_DIR = "mulval_rules"
# generate_rules(CSV_PATHS, OUT_DIR, domain_prefix=False, sep_subtech=True, force=True)
# recommended use: no Prefix


In [ ]:
# Example:
CSV_PATHS = ["enterprise-techniques.csv", "mobile-techniques.csv", "ics-techniques.csv"]
OUT_DIR = "mulval_rules"
generate_rules(CSV_PATHS, OUT_DIR, domain_prefix=False, sep_subtech=True, force=True)

# 💻 4. Create Data Structures for NLP-Pipeline

## 📈 Approach: Incremental Input Processing

* **Input:** Natural Language Rule Description <-- in Rule stubs (done)
* **Define Structure** Predicate Identification (initial predicate base); following initial Variable Extraction, Bound/Unbound Variable Classification
* **Fill Data Structurees:** Variable Extraction, Bound/Unbound Variable Classification, Rule creation; Predicate Updates


## 🧱 Initial Predicate Definition and Data Structures

**Predefined vs. Incremental**

* *Predefined Predicates* (Pros): Structured approach, Ensures consistency, Prevents ad-hoc predicate creation, Matches formal logic programming paradigms
* *Incremental Predicates* (Pros): Flexibility (on the Fly Predicate generation), Adapts to emerging rule complexities, Captures domain-specific nuances,     Allows organic rule development

**Conclusion:** Hybrid Approach

- Start with a core predicate set `predicates.P`
- Allow controlled predicate expansion
- Maintain a predicate registry with: Name, Arity, Description, Source/Origin, Usage contexts
- Ensure Human in the Loop: Additional Predicates have to be added manually (HITL Check)

### 🗄️ Data Structures: JSONL Records

We utilize a compact JSONL schema with three key pieces:

- Data 1 **Predicate record:**  one per name/arity, with a canonical arg schema and alias map
- Data 2 **Variable records:** global records that store bounded vs unbounded variable occurrences (so we are consistent across rules)

#### 📁 1. Predicate Record (global, canonical)

**Proposed Record Structure:**

```json
{"type":"predicate",
 "signature":"connects/3",
 "name":"connects",
 "arity":3,
 "args_schema":[
   {"pos":0,"name":"src","role":"host"},
   {"pos":1,"name":"dst","role":"host"},
   {"pos":2,"name":"port","role":"port"}
 ],
 "arg_aliases":{
   "src":["source","inet"],       // aliases seen in rules or NL
   "dst":["dest","internet"],     // <-- normalizes 'inet' vs 'internet'
   "port":["p","svc_port"]
 },
 "description":"Src can connect to Dst on Port.",
 "origin":"predicates.P",
 "status":"core",                // status is meta (for metadata predicates) and core for core logic rules
 "tags":["network"]}
```

- `args_schema` defines canonical positions and names.
- `arg_aliases` collects surface forms you encounter anywhere (rules or NL).


#### 📁 2. Global Variable/argument occurrence records

Produce additional records in the form of an append-only JSONL stores for variables 

**Canonical Concept Record**
- 'var_canonical.jsonl' is a sematic dictionary for variables. "What are the resusable, conceptual categories of variables in the KB?"
It collects all seen variable surfaces and a "preferred" name. It serves as the single project-wide vocabulary for consistent variable naming across rules and LLM-generated content.


### 🔄 Initialize 1 - Predicate table (global, canonical)

Builds initial core and meta predicate JSONL from `initial_predicates.P`.

In [1]:
# Build core+meta predicate JSONL from initial_predicates.P

import json, os, re
from typing import List, Tuple, Dict

# ---------- Regexes ----------
COMMENT_RE   = re.compile(r'^\s*%+\s?(.*)$')
DIRECTIVE_RE = re.compile(r'^\s*:-\s*')  # e.g., ":- table foo/2."
# wrapper forms: primitive(host(_H))., derived(exposed(H,Port))., meta(technique(...)).
WRAP_RE      = re.compile(r'^\s*([a-z][A-Za-z0-9_]*)\s*\(\s*([a-z][A-Za-z0-9_]*)\s*\((.*)\)\s*\)\s*\.\s*$')
# plain heads: connects(Net,H,Port) :- ...  OR  connects(Net,H,Port).
HEAD_RE      = re.compile(r'^\s*([a-z][A-Za-z0-9_]*)\s*\((.*)\)\s*(?:\.:?|-:|:-|\.)\s*$')

def split_args(arg_str: str) -> List[str]:
    """Split arguments by commas, respecting nested parentheses."""
    args, cur, depth = [], [], 0
    for ch in arg_str:
        if ch == '(':
            depth += 1; cur.append(ch)
        elif ch == ')':
            depth = max(0, depth-1); cur.append(ch)
        elif ch == ',' and depth == 0:
            tok = ''.join(cur).strip()
            if tok: args.append(tok)
            cur = []
        else:
            cur.append(ch)
    tail = ''.join(cur).strip()
    if tail: args.append(tail)
    return args

def normalize_arg_name(token: str, pos: int) -> str:
    """Turn a Prolog variable or atom into a canonical arg name; fallback arg{pos}."""
    tok = token.strip()
    # variable (starts with _ or capital)
    if re.match(r'^[_A-Z]', tok):
        base = tok.lstrip('_')
        base = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', base)
        base = re.sub(r'[^a-z0-9_]', '_', base, flags=re.I).strip('_').lower()
        if base: return base
    # atom/const (lowercase)
    if re.match(r'^[a-z]', tok):
        base = re.sub(r'[^a-z0-9_]', '_', tok, flags=re.I).strip('_').lower()
        if base: return base
    return f"arg{pos}"

def extract_predicates_with_comments(lines: List[str]):
    """
    Yield tuples:
      (layer, name, arity, arg_tokens, description)
    where layer ∈ {"core","meta"}.
    - Pulls inner predicate for wrapper forms (primitive/derived/meta).
    - Captures the closest preceding contiguous % comment block as description.
    - Skips directives (:- table ...).
    """
    current_cblock: List[str] = []
    for raw in lines:
        line = raw.rstrip()

        # normalize non-breaking spaces etc.
        line = line.replace("\u00a0", " ")

        # accumulate contiguous % comments
        cm = COMMENT_RE.match(line)
        if cm:
            current_cblock.append(cm.group(1).strip())
            continue

        # skip directives
        if DIRECTIVE_RE.match(line):
            if current_cblock: current_cblock = []
            continue

        # wrapper first: primitive(...). / derived(...). / meta(...).
        mwrap = WRAP_RE.match(line)
        if mwrap:
            wrapper = mwrap.group(1)          # primitive | derived | meta | ...
            inner_name = mwrap.group(2)
            inner_args = mwrap.group(3)
            args = split_args(inner_args)
            desc = " ".join(current_cblock).strip()
            current_cblock = []
            layer = "meta" if wrapper == "meta" else "core"
            yield (layer, inner_name, len(args), args, desc)

            # also emit meta/1 wrapper predicate once (no args list beyond the one term)
            if wrapper == "meta":
                # meta/1 is a separate predicate in the meta layer
                yield ("meta", "meta", 1, ["fact"], "Wrapper for metadata predicates.")
            continue

        # plain heads: name(args) :- ...    OR   name(args).
        m = HEAD_RE.match(line)
        if m:
            name, inner = m.group(1), m.group(2)
            args = split_args(inner)
            desc = " ".join(current_cblock).strip()
            current_cblock = []
            yield ("core", name, len(args), args, desc)
            continue

        # any other content breaks the comment block
        if current_cblock:
            current_cblock = []

def build_pred_records(found_iter) -> (Dict[str,Dict], Dict[str,Dict]):
    """
    Build two dicts of predicate records (core, meta), deduped by signature.
    First-seen wins for description/arg names.
    """
    core: Dict[str, Dict] = {}
    meta: Dict[str, Dict] = {}

    def ensure(rec_map: Dict[str, Dict], name: str, arity: int, args_tokens: List[str], desc: str, status: str):
        sig = f"{name}/{arity}"
        if sig in rec_map:
            return
        args_schema = [{"pos": i, "name": normalize_arg_name(tok, i), "role": "unknown"}
                       for i, tok in enumerate(args_tokens)]
        rec_map[sig] = {
            "type": "predicate",
            "signature": sig,
            "name": name,
            "arity": arity,
            "args_schema": args_schema,
            "arg_aliases": {},
            "description": desc or "",
            "origin": "initial_predicates.P",
            "status": status,
            "tags": []
        }

    for layer, name, arity, args_tokens, desc in found_iter:
        if layer == "meta":
            ensure(meta, name, arity, args_tokens, desc, status="meta")
        else:
            ensure(core, name, arity, args_tokens, desc, status="core")

    return core, meta

def write_jsonl(path: str, records: Dict[str, Dict]) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as out:
        for sig in sorted(records.keys()):
            out.write(json.dumps(records[sig], ensure_ascii=False) + "\n")

def build_core_and_meta_jsonl(input_path: str, out_core: str, out_meta: str):
    with open(input_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    found_iter = extract_predicates_with_comments(lines)
    core, meta = build_pred_records(found_iter)

    write_jsonl(out_core, core)
    write_jsonl(out_meta, meta)

    print(f"[OK] Wrote {len(core)} core predicate records → {out_core}")
    print(f"[OK] Wrote {len(meta)} meta predicate records  → {out_meta}")

# ---- Set paths and run ----
INPUT_PATH  = "initial_predicates.P"     # change if needed
OUTPUT_CORE = "nlp_inputs/predicates_core.jsonl"    # core domain predicates
OUTPUT_META = "nlp_inputs/predicates_meta.jsonl"    # meta layer predicates


In [64]:
build_core_and_meta_jsonl(INPUT_PATH, OUTPUT_CORE, OUTPUT_META)

[OK] Wrote 63 core predicate records → nlp_inputs/predicates_core.jsonl
[OK] Wrote 9 meta predicate records  → nlp_inputs/predicates_meta.jsonl


### 🔄 Initialize 2 Variable–Argument Canonicalization record (Initialization from Predicates)

This step builds a global variable concept dictionary (`var_canonical.jsonl`) directly from the predicate registries (`predicates_core.jsonl`, `predicates_meta.jsonl`).

Each record represents a semantic concept (like host, user, port, network) derived from argument roles defined in your predicates.
It serves as the single project-wide vocabulary for consistent variable naming across rules and LLM-generated content.

**Key points:** 
- Extracts argument roles from all predicates (core + meta).
- Maps roles to canonical concept keys ("host", "port", "network", etc.).
- Seeds each concept with a preferred surface form (e.g., "Host") and minimal role hint.
- Produces a lean, append-only JSONL file — later expanded automatically when new variable surfaces appear in generated rules.

This file becomes the naming and concept reference for your LLM rule generation pipeline, ensuring every new or updated rule stays semantically aligned with the project's existing fact base. 


In [4]:
# Build var_canonical.jsonl from predicates_core/meta.jsonl

import json, os
from collections import defaultdict
from typing import Dict, List, Any

PRED_CORE_JSONL = "nlp_inputs/predicates_core.jsonl"
PRED_META_JSONL = "nlp_inputs/predicates_meta.jsonl"
VAR_CANON_JSONL = "nlp_inputs/var_canonical.jsonl"

# --- IO helpers ---
def _load_jsonl(path: str) -> List[dict]:
    if not os.path.exists(path): return []
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def _append_jsonl(path: str, obj: Any) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def _titlecase_var(name: str) -> str:
    return "".join(p[:1].upper() + p[1:] for p in name.split("_") if p)

# --- Heuristics to infer a role if it's missing/unknown ---
NAME_TO_ROLE = {
    # network/host/port basics
    "src": "network", "source": "network", "net": "network", "network": "network",
    "dst": "host", "host": "host", "h": "host", "p": "port", "port": "port",
    # auth/identity
    "user": "user", "principal": "user", "acct": "user",
    "method": "service",
    # misc common
    "service": "service", "svc": "service", "proto": "service",
    "cve": "id", "id": "id", "url": "uri", "stix_id": "id",
    "level": "privilege", "priv": "privilege"
}

# Map roles to concept keys 
ROLE_TO_CONCEPT = {
    "host": "host",
    "port": "port",
    "network": "network",
    "user": "user",
    "service": "service",
    "id": "id",
    "uri": "uri",
    "privilege": "privilege",
    "category": "category",
    "text": "text",
    "annotation": "annotation",
    "unknown": "entity"  # fallback bucket
}

def _infer_role(name: str, role: str) -> str:
    if role and role != "unknown":
        return role
    key = (name or "").lower()
    return NAME_TO_ROLE.get(key, "unknown")

def build_var_canonical_from_predicates(core_path: str, meta_path: str, out_path: str):
    records = _load_jsonl(core_path) + _load_jsonl(meta_path)

    # concept -> {preferred, hints, slots:set[(sig,pos,name,role)]}
    concepts: Dict[str, Dict[str, Any]] = defaultdict(lambda: {
        "preferred": None,
        "hints": {},
        "slots": set()  # we won't persist slots (to keep lean), but we can use them to decide hints
    })

    for rec in records:
        if rec.get("type") != "predicate":
            continue
        sig = rec.get("signature")
        for arg in rec.get("args_schema", []):
            name = arg.get("name") or ""
            role = _infer_role(name, arg.get("role", "unknown"))
            concept = ROLE_TO_CONCEPT.get(role, "entity")

            # record a slot example internally (not persisted)
            concepts[concept]["slots"].add((sig, int(arg.get("pos", 0)), name, role))

            # seed preferred if unset
            if not concepts[concept]["preferred"]:
                concepts[concept]["preferred"] = _titlecase_var(concept)

            # seed hints (keep it minimal)
            concepts[concept]["hints"].setdefault("role", role)

    # Emit one append-only record per concept (no surfaces yet; we add them later during rule normalization)
    count = 0
    for concept, data in sorted(concepts.items()):
        _append_jsonl(out_path, {
            "type": "var_canonical",
            "key": concept,
            "surfaces": [data["preferred"]],  # seed with preferred; real surfaces get added later
            "preferred": data["preferred"],
            "hints": {"role": data["hints"].get("role", "unknown")}
        })
        count += 1

    print(f"[OK] Seeded {count} concepts → {out_path}")


[OK] Seeded 9 concepts → nlp_inputs/var_canonical.jsonl


In [ ]:
# ---- run ----
build_var_canonical_from_predicates(PRED_CORE_JSONL, PRED_META_JSONL, VAR_CANON_JSONL)

# ✍️ 5. LLM-assisted Rule Creation/Updating from Natural Language Attack Descriptions 

**📝 Inputs:**

**Initial/Incremental Predicate and Variable Base**
`PRED_CORE_JSONL = "nlp_inputs/predicates_core.jsonl"`    # core domain predicates
`PRED_META_JSONL = "nlp_inputs/predicates_meta.jsonl"`    # meta layer predicates
`VAR_CANON_JSONL = "nlp_inputs/var_canonical.jsonl"`      # global variable concept dictionary


**📝 LLM Pipeline:**

`AGENT_PROMPT = "nlp_inputs/systemprompt.md"`          # system prompt template

### 🤖 Natural Language Processing Pipeline: Variable Extraction, Bound/Unbound Variable Classification, Rule creation

- 1. The script scans a directory (recursively) for *.rule files, extracts the content.
- 2. LLM Pipeline: the an LLM is called and presented with a system prompt: rule the *.rule file, and nlp_inputs/predicates_core.jsonl, nlp_inputs/predicates_meta.jsonl and nlp_inputs/var_canonical.jsonl as context and creates the rules.
- 3. the *.rule file is copied to nlp_outputs/ and the rule is updated. 
- 4. If necessary, the script adds new variables and predicates to nlp_inputs/var_canonical.jsonl and nlp_inputs/predicates_core.jsonl


`nlp_inputs/systemprompt.md`

## System Prompt for LLM: MulVAL Rule Generation

You are an advanced reasoning language model designed to analyze natural language input and generate rules based on cybersecurity techniques related to MITRE ATT&CK. Your task is to interpret the provided *.rule file, utilizing it to create meaningful MulVAL rules interpreted by XSB Prolog.

### Context
You will receive the following inputs:

1. **Rule File**: A structured input containing details about a cybersecurity technique:
   - **technique_id**: Unique identifier for the cybersecurity technique.
   - **technique_name**: Name of the technique (e.g., Denial of Service).
   - **domain**: Specific area of application (e.g., ICS).
   - **tactic(s)**: Describes the tactic utilized (e.g., inhibit-response-function).
   - **platforms**: Relevant platforms (e.g., None).
   - **url**: Link for more detailed information about the technique.
   - **is_subtechnique**: Indicates if the technique is a sub-technique.
   - **stix_id**: Identifier in the Structured Threat Information Expression format.
   - **description**: Detailed explanation of the technique and its implications.
   - **generated_by**: Source of the rule generation.
   - **rule_stub**: A placeholder for a MulVAL/Prolog rule representing the attacker interaction.

2. **Additional Context Files**:
   - **predicates_core.jsonl**: A list of core predicates for rule generation.
   - **predicates_meta.jsonl**: Metadata associated with rules and predicates.
   - **var_canonical.jsonl**: Canonical variables and their meanings used in the context of the rules.

### Instructions
Using the information from the rule file alongside the predicates found in the JSON files, your task is to:

1. **Interpret the Description**: Analyze the provided content, extracting relevant information about the cybersecurity technique.
2. **Create a MulVAL Rule**: Based on the description, craft a MulVAL rule that captures the essence of the attack.

### Constraints
- Stick to existing predicates and only create new predicates if absolutely necessary.
- Utilize the canonical names from **var_canonical.jsonl** as arguments; only create new ones if absolutely necessary.
- The rules must encapsulate the attack description, carry a straightforward and understandable name, and be as simple as possible.

### Variable Guidelines
- **Bound Variables**: Use bound variables when you want to refer to specific instances or entities that have been previously defined or are known in the context. They are often used in conditions that need to match existing data. For example:
  ```prolog
  compromised_device(Host) :- 
      infected(Host).
  ```
- **Unbound Variables**: Use unbound variables when you are generalizing or when you need to capture a range of instances without specifying which entities are involved. They allow you to create more flexible rules. For example:
  ```prolog
  attack(Host) :-
      attacker_at(net),
      connects(net, Host, _Port).
  ```

### Output Format

Use the following output format:

```prolog
rule:
% ---- RULE LLM START (minimal) ----
    [Your MulVAL rule here]
    % ---- RULE LLM END ----
new_predicates:
new_vars:
```

Leave the `new_predicates` and `new_vars` empty if no new variables or predicates are created.

### Example Output:

```prolog
rule:
% ---- RULE LLM START (minimal) ----
    infected(Host) :-
        attacker_at(net),
        connects(net, Host, _Port).
    compromise(Host, user) :-
        infected(Host).
    % ---- RULE LLM END ----
new_predicates:
new_vars:
```

### ✍️ Script for LLM-assisted Rule Creation/Updating from Natural Language Attack Descriptions 

The script performs the following steps:

1. **Recursively scans a rules directory for `*.rule` files.**

2. **Loads contextual inputs:**
   - `nlp_inputs/systemprompt.md`
   - `nlp_inputs/predicates_core.jsonl`
   - `nlp_inputs/predicates_meta.jsonl`
   - `nlp_inputs/var_canonical.jsonl`

3. **Calls an LLM** with the system prompt, context, and the current `.rule` file to produce a minimal MulVAL rule in the required output format.

4. **Writes the updated rule** into `nlp_outputs/` (mirrored structure).

5. **Parses `new_predicates` / `new_vars`** and appends them (if any) to the JSONL inputs.

6. **Keeps everything append-only and safe** (backups/dirs).


In [3]:
# %% LLM-assisted Rule Creation/Updating (Notebook-friendly)

import os, re, json, textwrap, openai
from pathlib import Path
from typing import Dict, List, Any

# -----------------------
# Config (edit these)
# -----------------------
RULES_DIR   = Path("mulval_rules")          # where the *.rule files live
INPUTS_DIR  = Path("nlp_inputs")            # has systemprompt.md + JSONLs
OUTPUTS_DIR = Path("nlp_outputs")           # updated rules go here (mirrors layout)
MODEL_NAME  = "gpt-4o"                      # ignored if DRY_RUN=True
DRY_RUN     = False                         # set False to actually call an LLM (see call_llm)

# -----------------------
# Tiny JSONL helpers
# -----------------------
def load_jsonl(path: Path) -> List[dict]:
    if not path.exists(): return []
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def append_jsonl(path: Path, record: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

# -----------------------
# Inputs loader
# -----------------------
def load_system_prompt(inputs_dir: Path) -> str:
    p = inputs_dir / "systemprompt.md"
    return p.read_text(encoding="utf-8")

def load_context_files(inputs_dir: Path) -> Dict[str, Any]:
    return {
        "predicates_core": load_jsonl(inputs_dir / "predicates_core.jsonl"),
        "predicates_meta": load_jsonl(inputs_dir / "predicates_meta.jsonl"),
        "var_canonical":   load_jsonl(inputs_dir / "var_canonical.jsonl"),
    }

# -----------------------
# Rule parsing helpers
# -----------------------

HEADER_RE = re.compile(r'^\s*%\s*([\w\-\s]+?):\s*(.*)$')
START_STUB = "% ---- RULE STUB START"
END_STUB   = "% ---- RULE STUB END"
START_LLM  = "% ---- RULE LLM START"
END_LLM    = "% ---- RULE LLM END"

def parse_rule_file(text: str) -> Dict[str, Any]:
    meta = {}
    lines = text.splitlines()

    # header entries like: % technique_id: T0814
    for ln in lines:
        m = HEADER_RE.match(ln)
        if m:
            key = m.group(1).strip().lower().replace(" ", "_")
            val = m.group(2).strip()
            meta[key] = val

    # optional stub block (for context)
    stub_block = None
    if START_STUB in text and END_STUB in text:
        i = next(i for i,l in enumerate(lines) if START_STUB in l)
        js = [j for j,l in enumerate(lines) if END_STUB in l and j >= i]
        if js:
            j = js[0]
            stub_block = "\n".join(lines[i:j+1])

    return {"meta": meta, "rule_stub": stub_block, "text": text}

# -----------------------
# Prompt assembly
# -----------------------
def build_user_payload(rule_path: Path, parsed: Dict[str, Any], ctx: Dict[str, Any]) -> str:
    meta_json = json.dumps(parsed["meta"], ensure_ascii=False, indent=2)
    core_json = json.dumps(ctx["predicates_core"], ensure_ascii=False)
    meta_ctx  = json.dumps(ctx["predicates_meta"], ensure_ascii=False)
    var_json  = json.dumps(ctx["var_canonical"], ensure_ascii=False)

    return textwrap.dedent(f"""
    System Prompt is provided separately.

    ### Rule File: {rule_path}
    ```prolog
{parsed["text"]}
    ```

    ### Extracted Metadata (from header)
    ```json
{meta_json}
    ```

    ### Context: predicates_core.jsonl
    ```json
{core_json}
    ```

    ### Context: predicates_meta.jsonl
    ```json
{meta_ctx}
    ```

    ### Context: var_canonical.jsonl
    ```json
{var_json}
    ```

    ### Task
    Using the inputs above, produce the required output format:

    rule:
    % ---- RULE LLM START (minimal) ----
      [Your MulVAL rule here]
    % ---- RULE LLM END ----
    new_predicates:
    new_vars:
    """).strip()

# -----------------------
# LLM (stub or real)
# -----------------------
def call_llm(system_prompt: str, user_content: str, model: str, dry_run: bool) -> str:
    if dry_run:
        # Minimal placeholder that respects the output format
        return textwrap.dedent("""
        rule:
        % ---- RULE LLM START (minimal) ----
        infected(Host) :-
            attacker_at(net),
            connects(net, Host, _Port).
        compromise(Host, user) :-
            infected(Host).
        % ---- RULE LLM END ----
        new_predicates:
        new_vars:
        """).strip()

    # Real call: comment from here if you do not have an API key and openai installed
    
    from openai import OpenAI
    client = OpenAI()
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role":"system","content":system_prompt},
            {"role":"user","content":user_content},
        ],
        temperature=0.2,
    )
    return resp.choices[0].message.content.strip() 
    
    # Real call: comment to here if you do not have an API key and openai installed
    
    raise RuntimeError("Set DRY_RUN=False and uncomment OpenAI call to use a real model.")

# -----------------------
# Parse LLM output
# -----------------------
def parse_llm_output(text: str) -> Dict[str, str]:
    t = text.strip()

    # rule block: from "rule:" down to the END tag
    m_rule = re.search(r'(?s)rule:\s*(%.*?% ---- RULE LLM END ----)', t)
    rule_block = m_rule.group(1).strip() if m_rule else ""

    def section(label: str) -> str:
        m = re.search(rf'{label}:\s*(.*?)(?:(?:\n\w+:\s)|\Z)', t, flags=re.IGNORECASE | re.DOTALL)
        return (m.group(1).strip() if m else "")

    new_predicates = section("new_predicates")
    new_vars = section("new_vars")

    return {"rule_block": rule_block, "new_predicates": new_predicates, "new_vars": new_vars}

# -----------------------
# Registry updates (append-only)
# -----------------------
def maybe_append_new_predicates(pred_core_path: Path, block: str) -> int:
    if not block: return 0
    count = 0
    for raw in block.splitlines():
        line = raw.strip()
        if not line: continue
        if line.startswith("{"):
            try:
                rec = json.loads(line)
                if rec.get("type") == "predicate" and "signature" in rec:
                    append_jsonl(pred_core_path, rec); count += 1
                    continue
            except Exception:
                pass
        m = re.match(r'^([a-z][\w]*)/(\d+)(?:\s+(.*))?$', line)
        if m:
            name, arity, desc = m.group(1), int(m.group(2)), (m.group(3) or "")
            rec = {
                "type":"predicate","signature":f"{name}/{arity}","name":name,"arity":arity,
                "args_schema":[{"pos":i,"name":f"arg{i}","role":"unknown"} for i in range(arity)],
                "arg_aliases":{},"description":desc,"origin":"llm","status":"experimental","tags":[]
            }
            append_jsonl(pred_core_path, rec); count += 1
    return count

def maybe_append_new_vars(var_canon_path: Path, block: str) -> int:
    if not block: return 0
    count = 0
    for raw in block.splitlines():
        line = raw.strip()
        if not line: continue
        if line.startswith("{"):
            try:
                rec = json.loads(line)
                if rec.get("type") == "var_canonical" and "key" in rec:
                    append_jsonl(var_canon_path, rec); count += 1
                    continue
            except Exception:
                pass
        m = re.match(r'^([a-z][\w\-]*)\s*:\s*(.+)$', line, flags=re.I)
        if m:
            key = m.group(1).lower()
            surfaces = [s.strip() for s in m.group(2).split(",") if s.strip()]
            preferred = surfaces[0] if surfaces else None
            rec = {"type":"var_canonical","key":key,"surfaces":surfaces,"preferred":preferred,"hints":{"role":key}}
            append_jsonl(var_canon_path, rec); count += 1
    return count

# -----------------------
# Integrate LLM rule into file content
# -----------------------
def integrate_llm_rule(original_text: str, llm_rule_block: str) -> str:
    if not llm_rule_block:
        return original_text  # nothing to insert

    lines = original_text.splitlines()

    # Replace existing LLM block if present
    if START_LLM in original_text and END_LLM in original_text:
        i = next(i for i,l in enumerate(lines) if START_LLM in l)
        js = [j for j,l in enumerate(lines) if END_LLM in l and j >= i]
        if js:
            j = js[0]
            new_lines = lines[:i] + llm_rule_block.splitlines() + lines[j+1:]
            return "\n".join(new_lines)

    # Else insert right after stub if present, or append at EOF
    if START_STUB in original_text:
        s = next(s for s,l in enumerate(lines) if START_STUB in l)
        insert_at = s + 1
        new_lines = lines[:insert_at] + [""] + llm_rule_block.splitlines() + [""] + lines[insert_at:]
        return "\n".join(new_lines)

    return original_text.rstrip() + "\n\n" + llm_rule_block + "\n"

# -----------------------
# Notebook-friendly driver
# -----------------------
def process_one_rule(rule_path: Path) -> Dict[str, Any]:
    system_prompt = load_system_prompt(INPUTS_DIR)
    ctx = load_context_files(INPUTS_DIR)

    text = rule_path.read_text(encoding="utf-8")
    parsed = parse_rule_file(text)
    user_payload = build_user_payload(rule_path, parsed, ctx)

    llm_raw = call_llm(system_prompt, user_payload, model=MODEL_NAME, dry_run=DRY_RUN)
    llm_parsed = parse_llm_output(llm_raw)

    updated_text = integrate_llm_rule(text, llm_parsed["rule_block"])

    out_path = OUTPUTS_DIR / rule_path.relative_to(RULES_DIR)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(updated_text, encoding="utf-8")

    added_preds = maybe_append_new_predicates(INPUTS_DIR / "predicates_core.jsonl", llm_parsed["new_predicates"])
    added_vars  = maybe_append_new_vars(INPUTS_DIR / "var_canonical.jsonl", llm_parsed["new_vars"])

    return {
        "rule_path": str(rule_path),
        "out_path": str(out_path),
        "added_predicates": added_preds,
        "added_vars": added_vars,
        "llm_output_preview": llm_parsed["rule_block"][:200] + ("..." if len(llm_parsed["rule_block"])>200 else "")
    }

def process_all_rules() -> List[Dict[str, Any]]:
    RULES_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    results = []
    files = list(RULES_DIR.rglob("*.rule"))
    if not files:
        print(f"[WARN] No .rule files found under {RULES_DIR.resolve()}")
        return results
    print(f"[INFO] Processing {len(files)} rule(s) …")
    for rp in files:
        try:
            info = process_one_rule(rp)
            print(f"[OK] {info['rule_path']} → {info['out_path']}"
                  + (f" | +preds:{info['added_predicates']}" if info['added_predicates'] else "")
                  + (f" | +vars:{info['added_vars']}" if info['added_vars'] else ""))
            results.append(info)
        except Exception as e:
            print(f"[ERROR] {rp}: {e}")
    return results

# ---- Quick run helper (uncomment to execute now) ----
# results = process_all_rules()
# results[:3]


In [ ]:
# ---- Quick run helper (uncomment to execute now) ----
results = process_all_rules()
results[:3]


### Normalizing duplicate rule stubs

Some generated rules contain overlapping stub and LLM delimiters. This post-processing step removes duplicate markers while preserving the generated rule body.

```Prolog
% NOTE: Autogenerated stub. Adjust predicates/logic to your KB.
% technique_id: T0800
% technique_name: Activate Firmware Update Mode
% domain: ics
% tactic(s): inhibit-response-function
% platforms: None
% url: https://attack.mitre.org/techniques/T0800
% is_subtechnique: no
% stix_id: attack-pattern--19a71d1e-6334-4233-8260-b749cae37953
% description: Adversaries may activate firmware update mode on devices to prevent expected response functions from engaging in reaction to an emergency or process malfunction. For example, devices such as protection relays may have an operation mode designed for firmware installation. This mode may halt process monitoring and related functions to allow new firmware to be loaded. A device left in update mode may be placed in an inactive holding state if no firmware is provided to it. By entering and leaving a device in this mode, the adversary may deny its usual functionalities.
% generated_by: notebook_generate_rules

% ---- RULE STUB START (minimal) ----

% ---- RULE LLM START (minimal) ----
    firmware_update_mode_activated(Host) :-
        attacker_at(Network),
        connects(Network, Host, Port),
        ics_exposed_endpoint(Host, Port),
        role(Host, protection_relay).

    deny_functionality(Host) :-
        firmware_update_mode_activated(Host),
        defense_disabled(Host, process_monitoring).
% ---- RULE LLM END ----

potential_t0800(Host) :-
    attacker_at(net),
    connects(net, Host, _Port).

compromise(Host, user) :-
    potential_t0800(Host).
% ---- RULE STUB END ----
```

The normalization is applied after rule generation. Original rule files are backed up before rewriting so changes can be reviewed or reverted.

In [1]:
# %% Post-processing cleaner 
import re
from pathlib import Path
from typing import List

CLEAN_DIR = Path("nlp_outputs")   # change to "mulval_rules" if needed
BACKUP_EXT = ".bak"               # set to "" to disable backups

# --- Flexible marker regexes ---
_STUB_START_LINE = re.compile(r'^\s*%\s*----\s*RULE\s+STUB\s+START\b.*$', re.IGNORECASE | re.MULTILINE)
_LLM_END_LINE = re.compile(r'^\s*%\s*----\s*RULE\s+LLM\s+END\b.*$', re.IGNORECASE | re.MULTILINE)

def clean_rule_text(text: str) -> str:
    # 1) Remove any STUB START line
    text = re.sub(_STUB_START_LINE, '', text)

    # 2) Retain everything up to the last LLM END line while keeping the line itself
    last_llm_end = None
    for m in _LLM_END_LINE.finditer(text):
        last_llm_end = m
    if last_llm_end:
        text = text[:last_llm_end.end()]  # Keeps everything up to and including LLM END

    # 3) Cleanup trailing spaces and excessive newlines
    text = re.sub(r'[ \t]+$', '', text, flags=re.MULTILINE)  # Trim trailing spaces
    text = re.sub(r'\n{3,}', '\n\n', text).rstrip() + "\n"  # Reduce excessive newlines

    return text

def clean_rule_file(path: Path) -> bool:
    src = path.read_text(encoding="utf-8")
    cleaned = clean_rule_text(src)
    if cleaned == src:
        return False
    if BACKUP_EXT:
        path.with_suffix(path.suffix + BACKUP_EXT).write_text(src, encoding="utf-8")
    path.write_text(cleaned, encoding="utf-8")
    return True

def clean_rules_in_dir(base: Path) -> List[Path]:
    changed = []
    for p in base.rglob("*.rule"):
        try:
            if clean_rule_file(p):
                changed.append(p)
        except Exception as e:
            print(f"[ERROR] Failed to clean {p}: {e}")
    return changed

# ---- Run cleaner ----
changed = clean_rules_in_dir(CLEAN_DIR)
print(f"[OK] Cleaned {len(changed)} file(s).")
for p in changed[:10]:
    print("  -", p)
if len(changed) > 10:
    print(f"  ... and {len(changed)-10} more")

[OK] Cleaned 963 file(s).
  - nlp_outputs/T1105_ingress_tool_transfer.rule
  - nlp_outputs/T1492_stored_data_manipulation.rule
  - nlp_outputs/T1410_network_traffic_capture_or_redirection.rule
  - nlp_outputs/T1112_modify_registry.rule
  - nlp_outputs/T1559_inter-process_communication.rule
  - nlp_outputs/T0828_loss_of_productivity_and_revenue.rule
  - nlp_outputs/T1398_boot_or_logon_initialization_scripts.rule
  - nlp_outputs/T0824_i_o_module_discovery.rule
  - nlp_outputs/T1436_commonly_used_port.rule
  - nlp_outputs/T1116_code_signing.rule
  ... and 953 more


#### (if necessary): Restore Rules from Backup: Another utility

In [ ]:
# %% Simple restore script — overwrite .rule files from .rule.bak backups

from pathlib import Path
import shutil

RESTORE_DIR = Path("nlp_outputs")  # or "mulval_rules" if needed

def restore_backups(base: Path):
    baks = list(base.rglob("*.rule.bak"))
    if not baks:
        print(f"[INFO] No .rule.bak files found under {base.resolve()}")
        return

    for bak in baks:
        target = bak.with_suffix("")  # remove .bak → .rule
        shutil.copy2(bak, target)
        print(f"[RESTORED] {target.name} ← {bak.name}")

    print(f"[OK] Restored {len(baks)} file(s) from backups.")

# ---- run it ----
restore_backups(RESTORE_DIR)

#### (if necessary): Delete all backup rules (after restoring): yet another utility

In [ ]:
# %% Cleanup script: remove all non-.rule files in nlp_outputs and subdirectories
from pathlib import Path

BASE_DIR = Path("nlp_outputs")

def clean_non_rule_files(base: Path):
    if not base.exists():
        print(f"[WARN] Directory not found: {base}")
        return

    removed = 0
    for p in base.rglob("*"):
        if p.is_file() and p.suffix != ".rule":
            try:
                p.unlink()
                removed += 1
                print(f"[DEL] {p.relative_to(base)}")
            except Exception as e:
                print(f"[ERROR] Could not delete {p}: {e}")

    print(f"[OK] Removed {removed} non-.rule file(s) from {base}")

# ---- Run for nlp_outputs and its 'subtechniques' subdir ----
clean_non_rule_files(BASE_DIR)
clean_non_rule_files(BASE_DIR / "subtechniques")

### Add new predicates - store them in a seperate .P file

scan `"nlp_inputs/predicates_core.jsonl"`for added predicates and add them to a file. 

In [2]:
import json
from pathlib import Path
from typing import List, Dict, Any

PRED_CORE_JSONL = Path("nlp_inputs/predicates_core.jsonl")
OUT_P_FILE      = Path("nlp_outputs/additional_predicates_from_llm.P")  # change if you like

def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def _var_from_name(name: str, pos: int) -> str:
    """
    Turn arg schema name into a valid Prolog variable (predicate).
    Fallback to Arg{pos} if missing.
    """
    base = (name or f"arg{pos}").strip()
    # normalize to UpperCamel without non-word characters
    parts = [p for p in re_split(r"[^A-Za-z0-9]+", base) if p]
    if not parts:
        parts = [f"Arg{pos}"]
    up = "".join(p[:1].upper() + p[1:] for p in parts)
    if not up[0].isalpha():  # ensure starts with a letter
        up = f"A{up}"
    return up

def re_split(pattern: str, text: str) -> List[str]:
    import re
    return re.split(pattern, text)

def build_primitive_decl(rec: Dict[str, Any]) -> str:
    """
    Build a 'primitive(name(Args)).' line with a leading comment block.
    """
    name = rec["name"]
    arity = rec.get("arity", 0)
    args_schema = rec.get("args_schema", [])
    
    # Ensure args are ordered by pos
    args_schema = sorted(args_schema, key=lambda a: a.get("pos", 0))
    
    # Build variable list using the args_schema
    vars_list = [_var_from_name(args_schema[i].get("name"), i) for i in range(arity)]
    args_str = ", ".join(vars_list)

    # Compose a compact comment
    desc = (rec.get("description") or "").strip()
    sig  = rec.get("signature", f"{name}/{arity}")
    origin = rec.get("origin", "")
    status = rec.get("status", "")
    tags = rec.get("tags", [])
    tag_str = f" [tags: {', '.join(tags)}]" if tags else ""

    comment_lines = []
    comment_lines.append(f"% {sig}  (origin: {origin}, status: {status}){tag_str}")
    if desc:
        comment_lines.append(f"% {desc}")

    return "\n".join(comment_lines) + f"\nprimitive({name}({args_str})).\n"


def export_llm_predicates_to_P(pred_core_jsonl: Path, out_path: Path) -> int:
    recs = load_jsonl(pred_core_jsonl)
    # Filter: origin == "llm", skip meta
    llm_preds = [
        r for r in recs
        if r.get("type") == "predicate"
        and r.get("origin") == "llm"
        and r.get("status") != "meta"
    ]
    # Deduplicate by signature (last one wins)
    by_sig: Dict[str, Dict[str, Any]] = {}
    for r in llm_preds:
        sig = r.get("signature") or f"{r['name']}/{r.get('arity',0)}"
        by_sig[sig] = r

    # Sort by signature for stable output
    ordered = [by_sig[s] for s in sorted(by_sig.keys())]

    # Write .P file
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", encoding="utf-8") as f:
        f.write("% ---- Auto-exported LLM-origin predicates (primitive declarations) ----\n")
        f.write("% Source: nlp_inputs/predicates_core.jsonl\n\n")
        for rec in ordered:
            f.write(build_primitive_decl(rec))
            f.write("\n")

    return len(ordered)

# Run export
count = export_llm_predicates_to_P(PRED_CORE_JSONL, OUT_P_FILE)
print(f"[OK] Wrote {count} predicate(s) → {OUT_P_FILE}")

[OK] Wrote 99 predicate(s) → nlp_outputs/additional_predicates_from_llm.P


## Function

Reads each predicate declared in ADDED_PREDICATES_P
Scans all *.rule files under RULES_DIR (including subdirs)
Extracts the real variable names used at each argument position for those predicates
Rewrites ADDED_PREDICATES_P in place, replacing Arg0, _Arg1, … with the most common variable surface from the rules (preserves leading _)
Creates a .bak backup next to the original file

In [3]:
# %% Update additional_predicates_from_llm.P with real arg names learned from .rule files (FIXED) 

import re, collections
from pathlib import Path
from typing import Dict, List, Tuple

# --- Config ---
ADDED_PREDICATES_P = Path("nlp_outputs/additional_predicates_from_llm.P")
RULES_DIR          = Path("nlp_outputs")   # includes subdirs
BACKUP_EXT         = ".bak"

# ---------- Helpers ----------
def split_args(s: str) -> List[str]:
    args, cur, depth = [], [], 0
    for ch in s:
        if ch == '(':
            depth += 1; cur.append(ch)
        elif ch == ')':
            depth = max(0, depth-1); cur.append(ch)
        elif ch == ',' and depth == 0:
            tok = ''.join(cur).strip()
            if tok: args.append(tok)
            cur = []
        else:
            cur.append(ch)
    tail = ''.join(cur).strip()
    if tail: args.append(tail)
    return args

def is_var(tok: str) -> bool:
    t = tok.strip()
    return bool(t) and (t[0].isupper() or t[0] == "_")

def is_placeholder(tok: str) -> bool:
    return bool(re.match(r'^_?[Aa]rg\d+$', tok.strip()))

# Robust decl parser (tolerates whitespace, allows inline comments)
COMMENT_LINE_RE = re.compile(r'^\s*%')
DECL_BLOCK_RE = re.compile(
    r'(primitive|derived)\s*\(\s*([a-z][\w]*)\s*\((.*?)\)\s*\)\s*\.\s*(?:%.*)?',
    re.IGNORECASE | re.DOTALL
)

# For single-line rewriting
DECL_LINE_RE = re.compile(
    r'^\s*(primitive|derived)\s*\(\s*([a-z][\w]*)\s*\((.*?)\)\s*\)\s*\.\s*(?:%.*)?$',
    re.IGNORECASE
)

# Call matcher inside .rule files
CALL_RE = re.compile(r'([a-z][A-Za-z0-9_]*)\s*\((.*?)\)', re.DOTALL)

def load_added_signatures(pfile: Path) -> Dict[str, int]:
    """
    Parse primitive/derived declarations in ADDED_PREDICATES_P.
    Returns { 'name/arity': arity }.
    """
    sigs: Dict[str, int] = {}
    if not pfile.exists():
        print(f"[WARN] Missing {pfile}")
        return sigs

    raw = pfile.read_text(encoding="utf-8").replace("\u00a0", " ")
    lines = [ln for ln in raw.splitlines() if not COMMENT_LINE_RE.match(ln)]
    text = "\n".join(lines)

    for m in DECL_BLOCK_RE.finditer(text):
        name = m.group(2)
        args_block = m.group(3)
        args = split_args(args_block)
        sigs[f"{name}/{len(args)}"] = len(args)
    if not sigs:
        print(f"[WARN] No predicates found in {pfile}")
    else:
        print(f"[INFO] Found {len(sigs)} predicate declaration(s) in {pfile}")
    return sigs

def collect_rule_varnames(rules_dir: Path, target_sigs: Dict[str,int]) -> Dict[str, Dict[int, collections.Counter]]:
    """
    For each target signature, count variable names per argument position across all .rule files.
    Returns: sig -> pos -> Counter({surfaceVar: count})
    """
    counts: Dict[str, Dict[int, collections.Counter]] = {}
    for rp in rules_dir.rglob("*.rule"):
        text = rp.read_text(encoding="utf-8").replace("\u00a0", " ")
        # strip full-line comments
        body = "\n".join(ln for ln in text.splitlines() if not ln.strip().startswith("%"))
        for m in CALL_RE.finditer(body):
            name, inner = m.group(1), m.group(2)
            args = split_args(inner)
            sig = f"{name}/{len(args)}"
            if sig not in target_sigs:
                continue
            for pos, a in enumerate(args):
                a = a.strip()
                if not is_var(a) or is_placeholder(a):
                    continue
                counts.setdefault(sig, {}).setdefault(pos, collections.Counter())[a] += 1
    return counts

def choose_surface(cnt: collections.Counter) -> str:
    """Pick the most frequent variable surface (e.g., 'Host')."""
    if not cnt: return ""
    return cnt.most_common(1)[0][0]

def rewrite_added_file(pfile: Path, var_counts: Dict[str, Dict[int, collections.Counter]]) -> Tuple[int,int]:
    """
    Replace Arg0/Arg1... in ADDED_PREDICATES_P with most common var surfaces from rules.
    Returns (lines_changed, decls_touched).
    """
    if not pfile.exists():
        print(f"[WARN] Missing {pfile}")
        return (0,0)

    original = pfile.read_text(encoding="utf-8").replace("\u00a0", " ")
    out_lines: List[str] = []
    lines_changed = 0
    decls_touched = 0

    for ln in original.splitlines():
        m = DECL_LINE_RE.match(ln)
        if not m:
            out_lines.append(ln)
            continue

        kind, name, inner = m.group(1), m.group(2), m.group(3)
        args = split_args(inner)
        sig = f"{name}/{len(args)}"
        posmap = var_counts.get(sig, {})

        new_args = []
        replaced = False
        for i, a in enumerate(args):
            a_stripped = a.strip()
            lead = "_" if a_stripped.startswith("_") else ""
            if is_placeholder(a_stripped):
                surf = choose_surface(posmap.get(i, collections.Counter()))
                if surf:
                    # preserve leading underscore if placeholder had it
                    if lead and not surf.startswith("_"):
                        new_args.append(lead + surf)
                    else:
                        new_args.append(surf)
                    replaced = True
                else:
                    new_args.append(a)  # no observations → keep placeholder
            else:
                new_args.append(a)

        if replaced:
            decls_touched += 1
            lines_changed += 1
            out_lines.append(f"{kind}({name}({', '.join(new_args)})).")
        else:
            out_lines.append(ln)

    if lines_changed:
        if BACKUP_EXT:
            pfile.with_suffix(pfile.suffix + BACKUP_EXT).write_text(original, encoding="utf-8")
        pfile.write_text("\n".join(out_lines) + ("\n" if not original.endswith("\n") else ""), encoding="utf-8")

    return (lines_changed, decls_touched)

# ---- Run ----
target_sigs = load_added_signatures(ADDED_PREDICATES_P)
if not target_sigs:
    print(f"[WARN] No predicates found in {ADDED_PREDICATES_P}.")
else:
    counts = collect_rule_varnames(RULES_DIR, target_sigs)
    lines_changed, decls_touched = rewrite_added_file(ADDED_PREDICATES_P, counts)
    print(f"[OK] Updated {ADDED_PREDICATES_P}: {lines_changed} line(s) changed across {decls_touched} declaration(s).")


[INFO] Found 99 predicate declaration(s) in nlp_outputs/additional_predicates_from_llm.P
[OK] Updated nlp_outputs/additional_predicates_from_llm.P: 98 line(s) changed across 98 declaration(s).


## Final interaction rule file generation

Now that the core predicates and the single rule files have been created, we group them in one single file to run MulVAL. This is achieved with the `generate_full_kb.py` script.
The script reads the json files that contain the primitive predicates in `predicates_core.jsonl` and `additional_predicates_from_llm.P`, writes them in a Datalog file by parsing and declaring every predicate as primitive in correct format.
Tabling for all the predicates is applied to prevent from infinite loops when running the engine.
Then the script generates an interaction rule for each rule in the `nlp_outputs` folder, wraps its meta data and desctiption in the rule and writes everything in the `full_interaction_rules.P` file inside `kb`.
